# CP2 Week 2 -- Dict Analytics

**Course:** Computer Programming 2 (CP2)
**Prerequisites:** CP1 complete, Week 1 schema validation
**Focus:** counters, histograms, group summaries, collections.Counter

## Learning Objectives
- Use dictionaries as counters and accumulators
- Build group summaries (like SQL GROUP BY)
- Create text-based histograms
- Use `collections.Counter` for efficient counting
- Apply these patterns to your v2 pipeline analytics

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: Dictionaries as Counters

One of the most common patterns in data analysis is **counting things**. How many times does each value appear? Dictionaries are perfect for this.

Think of it like a tally sheet:
- You see "cat" -> make a tally mark next to "cat"
- You see "dog" -> make a tally mark next to "dog"
- You see "cat" again -> add another tally mark next to "cat"

In Python, the dictionary IS the tally sheet, and the values ARE the counts.

### Method 1: Manual Counting (the basic way)

In [ ]:
# Manual counting with if/else
animals = ["cat", "dog", "cat", "bird", "dog", "cat", "fish", "dog"]

counts = {}
for animal in animals:
    if animal in counts:
        counts[animal] += 1      # already seen: add 1
    else:
        counts[animal] = 1        # first time: start at 1

print("Counts:", counts)
print("Most cats:", counts["cat"])

**Expected Output:**
```
Counts: {'cat': 3, 'dog': 3, 'bird': 1, 'fish': 1}
Most cats: 3
```

### Method 2: Using .get() (cleaner)

In [ ]:
# Cleaner counting with .get()
animals = ["cat", "dog", "cat", "bird", "dog", "cat", "fish", "dog"]

counts = {}
for animal in animals:
    counts[animal] = counts.get(animal, 0) + 1
    # .get(key, default) returns the value if key exists,
    # otherwise returns the default (0)

print("Counts:", counts)

**Expected Output:**
```
Counts: {'cat': 3, 'dog': 3, 'bird': 1, 'fish': 1}
```

### Why This Matters

The `.get(key, default)` pattern is one of the most useful Python idioms. You will use it constantly in data processing. It avoids the if/else check and makes your code shorter and more readable.

In your v2 pipeline, you will count: drop reasons, status codes, categories, error types, and more.

### Method 3: collections.Counter (the best way)

In [ ]:
from collections import Counter

animals = ["cat", "dog", "cat", "bird", "dog", "cat", "fish", "dog"]

counts = Counter(animals)
print("Counter:", counts)
print("Type:", type(counts))
print()

# Counter has special methods:
print("Most common 2:", counts.most_common(2))
print("Cat count:", counts["cat"])
print("Missing key:", counts["elephant"])  # returns 0, no error!
print()

# You can add more data
more = Counter(["cat", "cat", "elephant"])
combined = counts + more
print("Combined:", combined)

**Expected Output:**
```
Counter: Counter({'cat': 3, 'dog': 3, 'bird': 1, 'fish': 1})
Type: <class 'collections.Counter'>

Most common 2: [('cat', 3), ('dog', 3)]
Cat count: 3
Missing key: 0

Combined: Counter({'cat': 5, 'dog': 3, 'bird': 1, 'fish': 1, 'elephant': 1})
```

### Common Mistakes: Counting

**Mistake 1:** Forgetting to initialize the count: `counts[animal] += 1` crashes if key does not exist.

**Fix:** Use `counts[animal] = counts.get(animal, 0) + 1` or use `Counter`.

**Mistake 2:** Using a list to count -- iterating the list each time to find the item is O(n) per lookup.

**Fix:** Dicts give O(1) lookup. Use a dict or Counter for counting.

**Mistake 3:** Not knowing about Counter -- writing 5 lines when 1 line does the job.

**Fix:** `from collections import Counter; counts = Counter(items)` -- done!


### Try It Yourself

Count the frequency of each letter in the string below. Use whichever method you prefer.

In [ ]:
# Try it: count letter frequencies
text = "abracadabra"
# TODO: count each letter and print the result


---
## Part 2: Counting Patterns in Data

In your v2 pipeline, you will often need to count categories in your data. Here is how to count values from a specific column in a list of dicts:

In [ ]:
from collections import Counter

# Pipeline data with a "status" column
data = [
    {"id": 1, "value": "25", "status": "ok"},
    {"id": 2, "value": "30", "status": "ok"},
    {"id": 3, "value": "",   "status": "missing"},
    {"id": 4, "value": "abc","status": "error"},
    {"id": 5, "value": "15", "status": "ok"},
    {"id": 6, "value": "-5", "status": "out_of_range"},
    {"id": 7, "value": "50", "status": "ok"},
    {"id": 8, "value": "",   "status": "missing"},
]

# Count statuses
status_counts = Counter(row["status"] for row in data)
print("Status counts:")
for status, count in status_counts.most_common():
    pct = count / len(data) * 100
    print("  " + status + ": " + str(count) + " (" + str(round(pct, 1)) + "%)")

**Expected Output:**
```
Status counts:
  ok: 4 (50.0%)
  missing: 2 (25.0%)
  error: 1 (12.5%)
  out_of_range: 1 (12.5%)
```

### Counting Drop Reasons

When your cleaning pipeline drops rows, you should count WHY each row was dropped. This is essential for your data quality report.

In [ ]:
def count_drop_reasons(data, numeric_columns, value_ranges):
    """Categorize each row and count reasons for dropping.
    
    Returns:
        dict with "kept", "dropped", and "reasons" keys
    """
    reasons = Counter()
    kept = 0
    
    for row in data:
        drop_reason = None
        
        # Check for missing values
        for col in numeric_columns:
            val = row.get(col, "")
            if val is None or str(val).strip() == "":
                drop_reason = "missing_" + col
                break
        
        # Check for non-numeric values
        if not drop_reason:
            for col in numeric_columns:
                try:
                    float(row[col])
                except (ValueError, TypeError):
                    drop_reason = "non_numeric_" + col
                    break
        
        # Check for out-of-range values
        if not drop_reason:
            for col, (low, high) in value_ranges.items():
                if col in row:
                    val = float(row[col])
                    if val < low or val > high:
                        drop_reason = "out_of_range_" + col
                        break
        
        if drop_reason:
            reasons[drop_reason] += 1
        else:
            kept += 1
    
    return {
        "total": len(data),
        "kept": kept,
        "dropped": sum(reasons.values()),
        "reasons": dict(reasons.most_common()),
    }

# Test
data = [
    {"id": 1, "value": "25"},
    {"id": 2, "value": ""},
    {"id": 3, "value": "abc"},
    {"id": 4, "value": "50"},
    {"id": 5, "value": "150"},   # out of range
    {"id": 6, "value": "-10"},    # out of range
    {"id": 7, "value": "30"},
]

report = count_drop_reasons(data, ["value"], {"value": (0, 100)})
print("Total:", report["total"])
print("Kept:", report["kept"])
print("Dropped:", report["dropped"])
print("Reasons:")
for reason, count in report["reasons"].items():
    print("  " + reason + ": " + str(count))

**Expected Output:**
```
Total: 7
Kept: 3
Dropped: 4
Reasons:
  out_of_range_value: 2
  missing_value: 1
  non_numeric_value: 1
```

---
## Part 3: Group Summaries

Often you need to analyze data separately for each group -- like computing average temperature per city, or average score per category.

This is the same idea as SQL's `GROUP BY`. In Python, we use a dict of lists.

### Group-By Concept

```
Raw Data:                     Grouped:
  Cairo  32                    Cairo:  [32, 35]  -> mean=33.5
  Cairo  35                    Alex:   [28, 26]  -> mean=27.0
  Alex   28                    Luxor:  [40, 42]  -> mean=41.0
  Alex   26
  Luxor  40
  Luxor  42
```

In [ ]:
def group_summary(data, group_col, value_col):
    """Compute summary statistics grouped by a column.
    
    Like a simple GROUP BY in SQL.
    
    Args:
        data: list of dicts
        group_col: column to group by
        value_col: column to compute stats on
    
    Returns:
        dict: {group_name: {"count": N, "sum": X, "mean": Y, "min": A, "max": B}}
    """
    # Step 1: collect values per group
    groups = {}
    for row in data:
        group = row.get(group_col, "unknown")
        try:
            val = float(row.get(value_col, 0))
        except (ValueError, TypeError):
            continue  # skip non-numeric
        
        if group not in groups:
            groups[group] = []
        groups[group].append(val)
    
    # Step 2: compute stats per group
    result = {}
    for group, vals in groups.items():
        result[group] = {
            "count": len(vals),
            "sum": round(sum(vals), 2),
            "mean": round(sum(vals) / len(vals), 2),
            "min": round(min(vals), 2),
            "max": round(max(vals), 2),
        }
    
    return result

# Test with city temperatures
data = [
    {"city": "Cairo", "temp": "32"},
    {"city": "Cairo", "temp": "35"},
    {"city": "Cairo", "temp": "33"},
    {"city": "Alex",  "temp": "28"},
    {"city": "Alex",  "temp": "26"},
    {"city": "Luxor", "temp": "40"},
    {"city": "Luxor", "temp": "42"},
    {"city": "Luxor", "temp": "38"},
]

result = group_summary(data, "city", "temp")
for city, stats in result.items():
    print(city + ": count=" + str(stats["count"])
          + ", mean=" + str(stats["mean"])
          + ", range=[" + str(stats["min"]) + "-" + str(stats["max"]) + "]")

**Expected Output:**
```
Cairo: count=3, mean=33.33, range=[32.0-35.0]
Alex: count=2, mean=27.0, range=[26.0-28.0]
Luxor: count=3, mean=40.0, range=[38.0-42.0]
```

### Example: Multi-Column Group Summary

In [ ]:
# Group summary with multiple value columns
def multi_summary(data, group_col, value_cols):
    """Compute stats for multiple columns, grouped."""
    results = {}
    
    for col in value_cols:
        results[col] = group_summary(data, group_col, col)
    
    return results

data = [
    {"dept": "eng", "salary": "70000", "years": "5"},
    {"dept": "eng", "salary": "85000", "years": "8"},
    {"dept": "sales", "salary": "60000", "years": "3"},
    {"dept": "sales", "salary": "55000", "years": "2"},
    {"dept": "sales", "salary": "75000", "years": "7"},
]

results = multi_summary(data, "dept", ["salary", "years"])
for col, groups in results.items():
    print(col + ":")
    for group, stats in groups.items():
        print("  " + group + ": mean=" + str(stats["mean"]))

**Expected Output:**
```
salary:
  eng: mean=77500.0
  sales: mean=63333.33
years:
  eng: mean=6.5
  sales: mean=4.0
```

### Try It Yourself

Use group_summary on your own data. Create a list of dicts with at least 10 rows, a category column, and a numeric column.

In [ ]:
# Try it: your own group summary
# TODO: create data and run group_summary


---
## Part 4: Building Histograms With Dicts

A **histogram** shows the distribution of values by grouping them into bins. You can build one using a dictionary where keys are bin labels and values are counts.

Before matplotlib (Week 5), we will make text-based histograms that work anywhere.

In [ ]:
def build_histogram(values, bin_width=10):
    """Build a text histogram using a dictionary.
    
    Args:
        values: list of numbers
        bin_width: width of each bin
    
    Returns:
        dict: {bin_label: count}
    """
    bins = {}
    for v in values:
        bin_start = int(v // bin_width) * bin_width
        label = str(bin_start) + "-" + str(bin_start + bin_width)
        bins[label] = bins.get(label, 0) + 1
    
    # Sort by bin start
    sorted_bins = dict(sorted(
        bins.items(),
        key=lambda x: int(x[0].split("-")[0])
    ))
    
    # Print text histogram
    max_count = max(sorted_bins.values())
    for label, count in sorted_bins.items():
        bar_len = int(count / max_count * 40)
        bar = "#" * bar_len
        print("  " + label.rjust(10) + " | " + bar + " (" + str(count) + ")")
    
    return sorted_bins

# Generate some data
import random
random.seed(42)
values = [random.gauss(50, 15) for _ in range(100)]
print("Distribution of 100 random values (mean=50, std=15):")
hist = build_histogram(values, bin_width=10)

**Expected Output:**
```
Distribution of 100 random values (mean=50, std=15):
      10-20 | ### (3)
      20-30 | ########## (10)
      30-40 | #################### (20)
      40-50 | ############################ (28)
      50-60 | ####################################### (22)
      60-70 | ############ (12)
      70-80 | #### (4)
      80-90 | # (1)
(approximate -- random seed may vary slightly)
```

### Histogram for Pipeline Data

In [ ]:
# Build histogram from pipeline data
data = [
    {"id": i, "value": str(round(random.gauss(50, 20), 1))}
    for i in range(50)
]

# Extract numeric values, skipping bad ones
values = []
for row in data:
    try:
        values.append(float(row["value"]))
    except (ValueError, TypeError):
        pass

print("Value distribution (" + str(len(values)) + " valid values):")
hist = build_histogram(values, bin_width=10)

### Key Takeaway

- Use `.get(key, default)` for safe dict access with defaults
- `collections.Counter` is the fastest way to count things
- Group summaries collect values per group, then compute stats
- Text histograms are a quick way to see distributions without matplotlib
- Always count drop reasons in your cleaning pipeline

### Debugging Tips: Dict Analytics

- KeyError means the key does not exist -- use .get() instead of []
- If Counter gives unexpected results, check that input values are the right type
- Empty groups usually mean the group_col has no matching values -- print data[0] to check
- If histogram bins look wrong, check bin_width and the range of your values

---
## Mini-Quiz

In [ ]:
# Q1: What does dict.get("key", 0) return if "key" does not exist?
# Answer: 

# Q2: What is the advantage of Counter over manual counting?
# Answer: 

# Q3: How would you count the number of rows per category in your pipeline?
# Answer: 

---
## Homework: 12 Exercises

### Review (1-4)

In [ ]:
# HW1: Count the frequency of each word in this sentence:
sentence = "the cat sat on the mat and the cat slept"
# Use Counter


In [ ]:
# HW2: Given this data, count how many rows have each status:
data = [
    {"id": 1, "status": "ok"},
    {"id": 2, "status": "error"},
    {"id": 3, "status": "ok"},
    {"id": 4, "status": "warning"},
    {"id": 5, "status": "ok"},
    {"id": 6, "status": "error"},
]


In [ ]:
# HW3: Write group_summary for this data, grouped by "category":
data = [
    {"category": "A", "score": "90"},
    {"category": "B", "score": "75"},
    {"category": "A", "score": "85"},
    {"category": "C", "score": "60"},
    {"category": "B", "score": "80"},
]


In [ ]:
# HW4: Explain in comments: why is counting drop reasons important
# for a data quality report?


### Practice (5-8)

In [ ]:
# HW5: Write a function that takes pipeline data and returns
# a "drop reason summary" with counts and percentages.


In [ ]:
# HW6: Build a histogram of the values in your project data.
# Choose an appropriate bin_width.


In [ ]:
# HW7: Write a function that computes the top-N most common values
# in a given column. Return as a list of (value, count) tuples.


In [ ]:
# HW8: Combine group_summary and histogram: for each group,
# print a mini-histogram of the values.


### Challenge (9-11)

In [ ]:
# HW9: Implement a "running counter" that updates counts as new
# data arrives (simulating streaming data). Use Counter.update().


In [ ]:
# HW10: Write a cross-tabulation function: given two columns,
# count all combinations (e.g., city x status).


In [ ]:
# HW11: Build a "data profile" function that automatically
# analyzes every column: counts for strings, stats for numbers.


### Mini-Project

In [ ]:
# HW12: Build a complete analytics dashboard (text-based) for
# your project data. It should display:
# - Row counts by category
# - Summary stats per group
# - Histogram of main numeric column
# - Top 5 most common values per string column
# Print everything in a readable format.


---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)